# 영화 정보 구조화 — LangChain 클라이언트 버전

영화 제목 입력 → **web_search 툴로 검색**(`bind_tools`) → **Structured Output**(`with_structured_output`)으로 공통 `Movie` 스키마에 맞춰 구조화한다.

OpenAI 버전과 **동일한 스키마/프롬프트**를 공유한다. 다른 점은 흐름을 **2단계 체인**으로 나눈 것.

## 1. 환경 설정

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트(schemas.py 가 있는 곳)를 import 경로에 추가
ROOT = Path.cwd()
if not (ROOT / "schemas.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

TITLE = "인터스텔라"  # ← 원하는 영화 제목으로 변경
MODEL = "gpt-5.4-nano"  # web_search + structured output 지원
EFFORT = "low"          # reasoning effort (low: 빠르고 저렴)
print("ROOT:", ROOT, "| TITLE:", TITLE, "| MODEL:", MODEL, "| effort:", EFFORT)

ROOT: C:\workspace\rnd-onboarding | TITLE: 인터스텔라 | MODEL: gpt-5.4-nano | effort: low


## 2. 1단계 — web_search 툴로 검색

내장 툴 `{"type": "web_search"}` 를 bind 하면 LangChain 이 자동으로 Responses API 로 전환해 서버측에서 검색을 수행한다. 결과 텍스트(findings)를 얻는다.

In [2]:
from langchain_openai import ChatOpenAI
import prompts
from schemas import Movie


def as_text(msg):
    """AIMessage.content(문자열 또는 블록 리스트)를 평범한 텍스트로 변환."""
    c = msg.content
    if isinstance(c, str):
        return c
    parts = [b.get("text", "") if isinstance(b, dict) else str(b) for b in c]
    return "\n".join(p for p in parts if p)


search_llm = ChatOpenAI(
    model=MODEL, use_responses_api=True, reasoning_effort=EFFORT
).bind_tools([{"type": "web_search"}])
search_msg = search_llm.invoke(
    f"영화 '{TITLE}' 의 개봉일, 배급사/제작사, 감독, 주요 출연진, 줄거리, 평점을 웹에서 찾아 정리해줘."
)
findings = as_text(search_msg)
print(findings)

## 영화 ‘인터스텔라(Interstellar, 2014)’

### 1) 개봉일(극장 개봉)
- **미국(미국 내 극장) : 2014년 11월 5일** ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interstellar_%28film%29))  
- **영국(극장) : 2014년 11월 7일** ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interstellar_%28film%29))  
- (참고) **미국 내 개봉 이전 일부 상영/프리미어 성격:** **2014년 10월 26일** ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interstellar_%28film%29))  
- Rotten Tomatoes 기준 “Theaters(와이드)” **2014년 11월 7일**로도 표기 ([rottentomatoes.com](https://www.rottentomatoes.com/m/interstellar_2014?adid=home_list3a))  

### 2) 배급사 / 제작사
- **배급(미국): Paramount Pictures** ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interstellar_%28film%29))  
- **배급(국제): Warner Bros. Pictures** ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interstellar_%28film%29))  
- **제작(Production companies):**
  - Paramount Pictures
  - Warner Bros. Pictures
  - Legendary Pictures
  - Syncopy
  - Lynda Obst Productions ([en.wikipedia.org](https://en.wikipedia.org/wiki/Interste

## 3. 2단계 — 구조화

검색 결과(findings)를 `with_structured_output(Movie)` 로 스키마에 맞춰 구조화한다.

In [3]:
struct_llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT).with_structured_output(Movie)

msgs = prompts.render("v2", TITLE, findings=findings)
movie = struct_llm.invoke([
    ("system", msgs["system"]),
    ("human", msgs["user"]),
])
print(movie.pretty())

🎬 인터스텔라  (2014-11-05)
⭐ 평점      : 8.7
🎥 감독      : 크리스토퍼 놀란(Christopher Nolan)
🏢 제작/배급 : Paramount Pictures
👥 출연      : 매튜 매커너히, 앤 해서웨이, 제시카 차스테인, 빌 어윈, 엘렌 버스틴
📖 줄거리    : 미래의 지구는 식량 위기와 환경 붕괴로 인해 더는 살기 어려운 상황이 됩니다. NASA의 물리학자 브랜드는 토성 근처의 웜홀을 통해 인류가 정착할 새로운 거처를 찾기 위한 계획을 세우고, 전직 조종사 쿠퍼와 연구팀을 파견합니다. 팀은 웜홀을 거쳐 은하를 탐사하며 인간의 생존을 위한 해답을 찾아 나섭니다.


## 4. 프롬프트 매니징 — v1 vs v2 비교

검색은 한 번만 하고(findings 재사용), 구조화 프롬프트만 v1/v2 로 바꿔 비교한다.

In [4]:
for v in prompts.list_versions():
    msgs = prompts.render(v, TITLE, findings=findings)
    m = struct_llm.invoke([
        ("system", msgs["system"]),
        ("human", msgs["user"]),
    ])
    print(f"=== prompt {v} ===")
    print(m.model_dump_json(indent=2))
    print()

=== prompt v1 ===
{
  "title": "인터스텔라 (Interstellar)",
  "release_date": "2014-11-05",
  "production": "크리스토퍼 놀란/파라마운트·워너브라더스·레전더리 픽처스 등 제작",
  "director": "크리스토퍼 놀란 (Christopher Nolan)",
  "cast": [
    "매튜 매커너히 (Cooper)",
    "앤 해서웨이 (Amelia Brand)",
    "제시카 차스테인 (Murph)",
    "빌 어윈 (TARS)",
    "엘렌 버스틴 (Murph, Older)"
  ],
  "synopsis": "미래의 지구는 식량 위기와 환경 붕괴로 더 이상 인간이 살기 어려운 상황에 놓인다. NASA의 물리학자 ‘브랜드’는 토성 근처의 웜홀을 통해 인류가 정착할 새로운 행성을 찾는 계획을 세우고, 전직 조종사 쿠퍼와 연구팀을 파견한다. 팀은 웜홀을 건너 은하를 탐사하며 생존을 위한 해답에 다가간다.",
  "rating": 7.4
}



=== prompt v2 ===
{
  "title": "인터스텔라",
  "release_date": "2014-11-07",
  "production": "Paramount Pictures",
  "director": "크리스토퍼 놀란",
  "cast": [
    "매튜 매커너히",
    "앤 해서웨이",
    "제시카 차스테인",
    "엘렌 버스틴",
    "빌 어윈"
  ],
  "synopsis": "미래의 지구는 식량 위기와 환경 붕괴로 인해 더는 살기 힘든 상황이 됩니다. NASA의 물리학자 브랜드는 토성 근처의 웜홀을 통해 인류가 정착할 “새로운 거처”를 찾기 위한 계획을 세우고, 전직 조종사 쿠퍼를 포함한 연구팀을 파견합니다. 팀은 웜홀을 거쳐 은하를 탐사하며 인간의 생존을 위한 해답을 찾아 나섭니다.",
  "rating": 8.7
}



## 5. 정리

- LangChain 은 **검색(`bind_tools`) → 구조화(`with_structured_output`)** 를 2단계 체인으로 구성.
- OpenAI 버전과 동일 스키마 → 결과 구조가 같다. 차이는 흐름의 추상화 방식.
- 2단계로 나누면 검색 결과를 재사용하거나 중간에 가공하기 쉽다(프롬프트 비교처럼).